# EEG Lab — Notebook 0: prepare a dataset

**Goal:** have an EEG data file ready, whether or not you managed to record with the OpenBCI headset on the day of the lab.

This notebook creates a file `eeg_data.csv` that mimics the output of an **OpenBCI Cyton** board (8 channels, 250 samples per second) during an **eyes-open / eyes-closed** experiment.

> 💡 If you have a real recording, jump to the end of the notebook to see how to use it instead.

---
### How do I run a cell?
Click inside the grey code cell, then press **Shift + Enter**. The result appears just below.

## 1. Import some tools

In Python you don't reinvent everything: you **import** toolboxes (called *libraries*). Here:
- `numpy`: to compute with arrays of numbers (nicknamed `np`).
- `pandas`: to handle spreadsheet-like tables (nicknamed `pd`).
- `matplotlib.pyplot`: to draw curves (nicknamed `plt`).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Set the experiment parameters

A **variable** is a label that holds a value. Read the line `fs = 250` as "the variable *fs* equals 250".

In [2]:
fs = 250            # sampling rate (samples per second)
n_channels = 8         # number of electrodes
block_duration = 30      # seconds for each condition (eyes open, then closed)

# Electrode names (10-20 system), a typical 8-channel montage
channel_names = ['Fp1','Fp2','C3','C4','P7','P8','O1','O2']

print("Total planned duration:", 2 * block_duration, "seconds")

Total planned duration: 60 seconds


## 3. Build a realistic EEG signal

An EEG signal is mostly a mix of waves at different frequencies plus noise. The star phenomenon of this lab: when you **close your eyes**, the **alpha** rhythm (~10 Hz) increases strongly, especially at the back of the head (electrodes O1, O2).

You don't need to understand everything in this cell: it just builds the data. We'll come back to it.

In [3]:
rng = np.random.default_rng(42)  # random seed: reproducible results

def block(condition, duration):
    """Create a signal block for one condition ('open' or 'closed')."""
    n = duration * fs
    t = np.arange(n) / fs                      # time axis in seconds
    signal = np.zeros((n, n_channels))
    for c in range(n_channels):
        # background noise (all frequencies)
        x = rng.normal(0, 8, n)
        # 10 Hz alpha rhythm: strong if eyes closed AND a back electrode
        back = channel_names[c].startswith(('O', 'P'))
        alpha_amp = 18 if (condition == 'closed' and back) else 3
        x += alpha_amp * np.sin(2*np.pi*10*t + rng.uniform(0, 6))
        # a bit of 50 Hz: electrical mains noise
        x += 4 * np.sin(2*np.pi*50*t)
        signal[:, c] = x
    return signal

sig_open   = block('open',   block_duration)
sig_closed = block('closed', block_duration)
signal = np.vstack([sig_open, sig_closed])    # stack the two blocks

print("Signal array shape:", signal.shape, "(rows = time, columns = channels)")

Signal array shape: (15000, 8) (rows = time, columns = channels)


## 4. Add a "marker" column

To know which part is *eyes open* (0) and *eyes closed* (1), we add a column of labels.

In [4]:
marker = np.concatenate([
    np.zeros(len(sig_open)),     # 0 = eyes open
    np.ones(len(sig_closed))     # 1 = eyes closed
])

print("Number of eyes-open samples  :", int((marker == 0).sum()))
print("Number of eyes-closed samples:", int((marker == 1).sum()))

Number of eyes-open samples  : 7500
Number of eyes-closed samples: 7500


## 5. Save to a CSV file

We put everything in a `pandas` table (a *DataFrame*) then write it to disk. This is the file the next notebooks will read.

In [5]:
df = pd.DataFrame(signal, columns=channel_names)
df['marker'] = marker
df.to_csv('eeg_data.csv', index=False)

print("File 'eeg_data.csv' saved.")
df.head()   # preview the first 5 rows

File 'eeg_data.csv' saved.


,Fp1,Fp2,C3,C4,P7,P8,O1,O2,marker
0,0.092000,-14.625484,8.987525,-6.547134,-4.944897,0.148880,-1.327193,4.959388,0.0
1,-7.252779,1.734556,10.285500,13.988153,-2.524806,1.201113,-0.761411,19.605560,0.0
2,5.398207,3.750631,8.400295,1.738137,5.435477,5.992507,2.404634,6.487170,0.0
3,2.183192,0.681534,2.229465,6.066458,-10.590069,6.060448,-3.369264,-1.654360,0.0
4,-22.248449,-9.149569,1.281958,-1.189225,-25.358966,-11.600134,-14.424075,0.147765,0.0


## 6. What about a real OpenBCI recording?

The OpenBCI GUI exports a `.txt`/`.csv` file with a few header comment lines (starting with `%`) followed by the channel columns. To load it:

```python
df = pd.read_csv('my_recording.txt', comment='%')
# The useful columns are usually 'EXG Channel 0' to 'EXG Channel 7'
```

What matters for the rest: ending up with an `n_samples × 8 channels` table. Keep in mind that **fs = 250 Hz** on the Cyton without the Daisy module.